# CREM: Porcentaje de Población Española
**Propósito**: Extrae la información sobre la evolución del porcentaje de población española según municipios, distritos y secciones censales a partir de la página oficial del CREM (Centro Regional de Estadística de Murcia), procesa y limpia los datos directamente en memoria y los almacena en una tabla Delta Lake en Spark.

In [1]:
import sys
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
# Definir la url de la web
url_base = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM2089//sec30.html"
table_name = "raw.crem.porcentaje_poblacion_espanola"

## Descarga y Parseo del Contenido HTML directamente en Memoria
Se realiza una petición HTTP para descargar el contenido HTML de la página oficial del CREM en memoria y procesarlo dinámicamente sin persistir archivos intermedios en el disco local.

In [3]:

# Hacer la solicitud
html = crem.get_html(url_base)

# Convertir HTML a BeautifulSoup object.
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")

if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# Obtener cabeceras (años)
thead = table.find("thead", id="cabeceraTablaDatos")
anos_columnas = []
for th in thead.find_all("th"):
    val = th.get_text(strip=True)
    if val:
        anos_columnas.append(val)

print(f"Años detectados: {anos_columnas}")

Años detectados: ['2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']


In [4]:
# Parsear filas de datos jerárquicos
tbody = table.find("tbody")
datos_filas = []

mapping_niveles = {
    1: "Región",
    2: "Municipio",
    3: "Distrito",
    4: "Sección"
}

for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue
    
    ul = th.find("ul")
    level_num = None
    if ul and ul.has_attr("class"):
        for c in ul["class"]:
            if c.startswith("level"):
                try:
                    level_num = int(c.replace("level", ""))
                except ValueError:
                    pass
                break
    
    label_raw = th.get_text(strip=True)
    label_clean = crem.clean_and_decode(label_raw)
    
    # Extraer código y nombre (ej. 'Abanilla - 30001' o 'Abanilla distrito 01 - 3000101')
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    if match:
        nombre = match.group(1).strip()
    else:
        nombre = label_clean

    if ',' in nombre: nombre=nombre.split(',')[1]+' '+nombre.split(',')[0]
    
    nivel = mapping_niveles.get(level_num, "Desconocido")
    
    tds = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            val_float = float(val_str) if val_str not in ("", "-", "..") else None
        except ValueError:
            val_float = None
        values.append(val_float)
    
    if len(values) == len(anos_columnas):
        registro = {
            "nombre": crem.normalize_name(nombre),
            "nivel": nivel
        }
        for a, val in zip(anos_columnas, values):
            registro[a] = val
        datos_filas.append(registro)

print(f"Filas parseadas con éxito: {len(datos_filas)}")

Filas parseadas con éxito: 1350


## Integración con PySpark y Delta Lake
Se crea un Spark Session, se inicializa el Spark DataFrame a partir del listado en memoria, se normaliza su esquema con las utilidades de TFM y se guarda como Delta Table en la ruta final.

In [5]:
# Inicializar sesión de Spark usando las utilidades de TFM
spark = tfm_utils.get_spark_session(app_name="CREM_Porcentaje_Poblacion_Espanola")

# Crear el Spark DataFrame a partir de la lista de registros estructurados
df_poblacion = (spark.createDataFrame(datos_filas)
                    .filter(F.col("nivel").like("Municipio"))
                    .drop("nivel")
               )

for anio in anos_columnas:
    df_poblacion=df_poblacion.withColumnRenamed(f"{anio}",f"{anio}_Porcentaje_Poblacion_Espanola")

# Normalizar las columnas del DataFrame de acuerdo al estilo del TFM (minúsculas, sin acentos, ordenadas)
df_poblacion_normalized = tfm_utils.normalize_df(df_poblacion)

# Mostrar los primeros registros de forma estructurada
tfm_utils.display(df_poblacion_normalized)

,2015_porcentaje_poblacion_espanola,2016_porcentaje_poblacion_espanola,2017_porcentaje_poblacion_espanola,2018_porcentaje_poblacion_espanola,2019_porcentaje_poblacion_espanola,2020_porcentaje_poblacion_espanola,2021_porcentaje_poblacion_espanola,2022_porcentaje_poblacion_espanola,2023_porcentaje_poblacion_espanola,nombre
0,88.2,88.2,88.8,87.5,86.9,86.4,85.0,83.9,83.8,abanilla
1,91.3,91.5,91.1,90.8,90.3,90.1,89.8,89.4,88.6,abaran
2,87.6,88.4,88.3,88.3,88.0,88.1,87.8,88.1,87.5,aguilas
3,99.2,99.3,99.0,99.1,98.9,98.3,98.1,97.8,97.4,albudeite
4,91.3,91.6,91.4,90.7,90.1,89.7,89.6,89.2,88.9,alcantarilla
5,95.1,95.9,93.5,93.3,92.3,92.1,92.2,92.5,93.0,aledo
6,85.6,86.9,87.0,86.8,86.3,86.2,85.5,85.3,85.3,alguazas
7,81.0,81.0,80.8,80.8,81.2,82.6,82.5,81.8,82.0,alhama_de_murcia
8,83.4,83.8,83.1,82.4,82.2,82.1,82.1,80.7,78.6,archena
9,77.9,77.5,77.4,76.0,74.6,74.5,74.4,75.2,76.1,beniel


In [6]:
# Ruta destino de la Delta Table
delta_path = tfm_utils.full_table_path(table_name)

print(f"Escribiendo {df_poblacion_normalized.count()} registros en la Delta Table: {table_name}")
print(f"Destino: {delta_path}")

# Escritura en modo overwrite
(df_poblacion_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)

print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en la Delta Table: raw.crem.porcentaje_poblacion_espanola
Destino: /tfm/delta_lake/raw/crem/porcentaje_poblacion_espanola
¡Escritura en Delta Lake completada con éxito!
